# 01 — Cache residual-stream activations

Runs Llama 3.1 8B forward passes over SelfAware prompts and saves `resid_post` at every layer, plus the generated responses for the judge.

**Inputs:** `data/selfaware/SelfAware.json` (topic-enriched).
**Outputs:** `responses.json`, `acts_layer_{i}.pt` (one per layer) in the results dir.

**Heavy step.** Run on Colab A100. `results_dir()` resolves to Google Drive on Colab, so caching survives a runtime restart — re-running this notebook skips already-cached shards.

In [ ]:
# --- Colab setup (skip these two lines when running locally) ---
# from google.colab import drive; drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/abstention-geometry/notebooks"
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
import json

from src.data import load_selfaware, build_prompts
from src.model import load_model, generate_responses, cache_residual_stream
from src.paths import results_dir

In [ ]:
DATA_PATH = '../data/selfaware/SelfAware.json'
RESULTS_DIR = results_dir()  # Drive on Colab, repo results/ locally; override with $RESULTS_DIR
print('RESULTS_DIR =', RESULTS_DIR)

## Load dataset + build prompts

In [ ]:
df = load_selfaware(DATA_PATH)
prompts = build_prompts(df)
print(f'{len(df)} questions | answerable rate: {df.answerable.mean():.2f}')
print(df.topic.value_counts())
print('\n--- example prompt ---\n' + prompts[0])

## Load model

In [ ]:
model = load_model()
print('loaded', model.cfg.model_name, '|', model.cfg.n_layers, 'layers | d_model', model.cfg.d_model)

## Generate responses

Greedy decoding. Save to `responses.json` for the judge in 02.

In [ ]:
responses = generate_responses(model, prompts, batch_size=8)
with open(RESULTS_DIR + 'responses.json', 'w') as f:
    json.dump({'question_id': df['question_id'].tolist(), 'response': responses}, f)
print('saved', RESULTS_DIR + 'responses.json')
print('\n--- example response ---\n' + responses[0])

## Cache residual stream

All 32 layers, last-token resid_post. Shard-checkpointed: re-running after a disconnect resumes from the last completed shard.

In [ ]:
cache_residual_stream(model, prompts, batch_size=8, save_dir=RESULTS_DIR)
print('done — acts_layer_0..%d.pt written to %s' % (model.cfg.n_layers - 1, RESULTS_DIR))